# Plotting Parrots

## Initialising codebase

In [ ]:
import sys
import os
from datetime import datetime as dt
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import plotly.express as px

### Import data

In [ ]:

beaufort_scale = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/beaufort_scale.csv')
birds = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/birds.csv', low_memory=False)
sea_states = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/sea_states.csv')
ships = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/ships.csv')

### Check if data imported correctly

In [ ]:
beaufort_scale.head()

In [ ]:
birds.head()

In [ ]:
sea_states.head()

In [ ]:
ships.head()

# Plotting by geographical location

### Purpose
Does **where** a count was taken change which bird groups were seen? This plot uses only latitude and longitude. Season is the plot's variable, so the seasons here are averaged out rather than compared.

### Step 1: group the species and pick the top 5
* `classify_bird` sorts each `species_common_name` into a group by keyword. The order of the checks matters: "storm-petrel" and "prion" are checked before "petrel", so those groups are not swallowed by Petrels. "Mollymawk" is the logbooks' name for the smaller albatrosses, so it counts as an albatross.
* Groups are ranked by the **number of counts they appear in** (unique `record_id`), not by total birds, so one huge flock cannot outweigh hundreds of separate sightings.
* Top 5: Petrels (7,890 counts), Albatrosses (7,812), Shearwaters (4,701), Prions (2,354), Gulls & terns (2,012).

In [ ]:
def classify_bird(name):
    name = str(name).lower()
    if "storm-petrel" in name or "storm petrel" in name:
        return "Storm petrels"
    elif "shearwater" in name:
        return "Shearwaters"
    elif "albatross" in name or "mollymawk" in name: # mollymawks are albatrosses
        return "Albatrosses"
    elif "prion" in name:
        return "Prions"
    elif "petrel" in name:
        return "Petrels"
    elif "fulmar" in name:
        return "Fulmars"
    elif "skua" in name:
        return "Skuas"
    elif "gull" in name or "tern" in name:
        return "Gulls & terns"
    elif "gannet" in name or "shag" in name:
        return "Gannets & shags"
    else:
        return "Other"

In [ ]:
birds_geo_loc = birds.copy()
birds_geo_loc["group"] = birds["species_common_name"].apply(classify_bird)
birds_geo_loc = birds_geo_loc.groupby(["record_id", "group"]).agg(count = ("count", "sum")).reset_index()
birds_geo_loc.head()

In [ ]:
top5_groups = (birds_geo_loc[birds_geo_loc["group"]!="Other"]
               .groupby("group")["record_id"].nunique()
               .sort_values(ascending=False)
               .head(5)
               .index)
print(top5_groups)

In [ ]:
birds_geo_loc = birds_geo_loc[birds_geo_loc["group"].isin(top5_groups)]

### Step 2: one row per count, with its location and quarter
* Keep only **full** counts (12,111 of 12,310). The 199 partial counts are shorter, so they are not comparable.
* Drop the 11 full counts with no coordinates, leaving **12,100**.
* Tag each count with its quarter of the year: Jan–Mar (3,457 counts), Apr–Jun (2,123), Jul–Sep (2,506), Oct–Dec (4,014).
* For each count and group, record True if the group was logged at least once. A group counts once per count, irregardless of however many birds were seen.

In [ ]:
# full censuses with coordinates, tagged with their quarter of the year
geo_loc = ships[ships["census_method"] == "full"][["record_id", "latitude", "longitude", "date"]].dropna()
geo_loc["quarter"] = pd.to_datetime(geo_loc["date"]).dt.quarter          # 1 = Jan–Mar ... 4 = Oct–Dec

In [ ]:
# one row per census, True/False for each group
present = birds_geo_loc.assign(present=True).pivot_table(index="record_id", columns="group", values="present",
                                                         aggfunc="any", fill_value=False)
cells = geo_loc.merge(present, left_on="record_id", right_index=True, how="left").fillna(False)

### Step 3: plot smoothed presence-rate maps with interactive hover
**Measure: presence rate**, the share of counts in a patch of sea that logged the group. We use a rate rather than raw sightings because ships surveyed some lanes far more often than others, so raw sightings would mostly map ship traffic.

1. **Grid.** Counts are binned into 0.1° cells (about 10 km).
2. **Smooth.** Both grids (all counts, and counts with the group) are blurred with a Gaussian filter (σ = 6 cells ≈ 0.6°), and the rate is one divided by the other. Each patch therefore draws on counts within roughly 60 km, which fills the gaps between neighbouring ship tracks without inventing data far from any ship.
3. **Weight the seasons equally.** The rate is worked out separately for each quarter, then the four are averaged. Without this, the Tasman Sea (39% of its counts in Jan–Mar but only 14% in Jul–Sep) would look like a summer place, and season would leak into a map that is meant to be about location.
4. **Only colour well-surveyed sea.** A cell is shown only if it had at least 6 counts nearby in **every** quarter. Blank sea means "not enough data", and not "no birds". About 27% of the map window fulfills the criteria, and the coloured areas follow the ship routes.
5. **Map window.** 147–180°E, 30–48.5°S keeps 86% of all full counts, including the Tasman crossings between Australia and New Zealand.
6. **Summary number.** The number in each map's corner is the median rate over all coloured sea, i.e. the rate at a typical surveyed spot.

The map is interactive: hover over a coloured patch to see its rate and how many counts it is based on.

In [ ]:
from scipy.ndimage import gaussian_filter
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from shapely.geometry import box
np.seterr(divide="ignore", invalid="ignore")

LON, LAT = (147, 180), (-48.5, -30) # map window zoomed in to cover only NZ surroundings and tasman sea between NZ and AU
lon_bins, lat_bins = np.arange(145, 182, 0.1), np.arange(-50.5, -28, 0.1)   # 0.1° grid, 2° spare around the window

def smooth_count(df, weights=None):
    """Count censuses in each 0.1° cell, then blur over ~0.6° (6 cells) so gaps between ship tracks fill in."""
    counts, _, _ = np.histogram2d(df["latitude"], df["longitude"], bins=[lat_bins, lon_bins], weights=weights)
    return gaussian_filter(counts, sigma=6) * 226

quarters = [cells[cells["quarter"] == q] for q in [1, 2, 3, 4]]
effort = [smooth_count(df) for df in quarters]
surveyed = np.all([e >= 6 for e in effort], axis=0)  # only keep sea with >= 6 censuses nearby in every quarter

rate = {}
for g in top5_groups:
    # % of censuses that logged g, per quarter, then the average of the four quarters (each season has equal weightage)
    per_quarter = [smooth_count(df, df[g].astype(float)) / e * 100 for df, e in zip(quarters, effort)]
    rate[g] = np.where(surveyed, np.mean(per_quarter, axis=0), np.nan)

lon_mid, lat_mid = lon_bins[:-1] + 0.05, lat_bins[:-1] + 0.05
rows = np.where((lat_mid >= LAT[0]) & (lat_mid <= LAT[1]))[0][::2]
cols = np.where((lon_mid >= LON[0]) & (lon_mid <= LON[1]))[0][::2]
inside = np.ix_(rows, cols)
nearby = sum(effort)[inside].round().astype("float32")
order = ["Albatrosses", "Petrels", "Prions", "Shearwaters", "Gulls & terns"]


In [ ]:
# trace coastline
land = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_land.geojson")
land = land.clip(box(LON[0] - 3, LAT[0] - 3, LON[1] + 3, LAT[1] + 3)).explode().simplify(0.02)
coast_x, coast_y = [], []
for poly in land:
    x, y = poly.exterior.xy
    coast_x += list(x) + [None]; coast_y += list(y) + [None]

# red color variations
reds = [[0, "#fef0d9"], [0.25, "#fdcc8a"], [0.5, "#fc8d59"], [0.75, "#e34a33"], [1, "#b30000"]]

fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.02, vertical_spacing=0.1,
                    subplot_titles=[f"<b>{g}</b>" for g in order])
for k, g in enumerate(order):
    r, c = k // 3 + 1, k % 3 + 1
    fig.add_trace(go.Contour(x=lon_mid[cols], y=lat_mid[rows], z=rate[g][inside].round().astype("float32"), customdata=nearby,
                             zmin=0, zmax=100, hoverongaps=False, line_width=0,
                             contours=dict(start=0, end=100, size=20), autocontour=False,
                             colorscale=reds, showscale=(k == 0), name=g,
                             hovertemplate=f"<b>{g}</b><br>lat %{{y:.1f}}, lon %{{x:.1f}}<br>"
                                           "Recorded in %{z:.0f}% of counts here<br>Based on %{customdata:.0f} nearby counts<extra></extra>",
                             colorbar=dict(orientation="h", x=0.835, y=0.1, len=0.3, thickness=14, tickvals=[0, 20, 40, 60, 80, 100],
                                           ticktext=["0%", "20%", "40%", "60%", "80%", "100%"],
                                           title=dict(text="% of 10-minute counts<br>that recorded the group", side="top"))),
                  row=r, col=c)
    fig.add_trace(go.Scatter(x=coast_x, y=coast_y, fill="toself", fillcolor="white", mode="lines",
                             line=dict(color="#5b6470", width=0.7), hoverinfo="skip", showlegend=False), row=r, col=c)

    ax = f"{k + 1 if k else ''}"
    fig.add_annotation(x=LON[1] - 0.6, y=LAT[0] + 0.6, xref=f"x{ax}", yref=f"y{ax}", xanchor="right", yanchor="bottom",
                       align="right", showarrow=False, bgcolor="rgba(255,255,255,0.85)", borderpad=4,
                       text=f"<span style='font-size:20px;color:#b30000'><b>{np.nanmedian(rate[g]):.0f}%</b></span><br>"
                            "<span style='font-size:10px;color:dimgrey'>typical spot</span>")
    fig.update_xaxes(range=LON, visible=False, row=r, col=c)
    fig.update_yaxes(range=LAT, visible=False, scaleanchor=f"x{k + 1 if k else ''}", scaleratio=1.3, row=r, col=c)

fig.add_annotation(x=157, y=-45.5, text="<i>Tasman Sea</i>", showarrow=False, font=dict(color="grey"), xref="x", yref="y")
fig.add_annotation(x=0.69, y=0.40, xref="paper", yref="paper", xanchor="left", align="left", showarrow=False,
                   font=dict(size=12, color="dimgrey"),
                   text="<b>How to read</b><br>"
                        "Darker = the group was recorded more often there.<br>"
                        "Big number = the typical (median) value across<br>all coloured sea. Hover a map for exact numbers.<br><br>"
                        "<b>Note:</b><br>"
                        "Coloured areas follow ship routes.<br>Blank sea had too few counts in at least one season.")
fig.update_layout(width=1050, height=640, plot_bgcolor="#e9eef2", paper_bgcolor="white", margin=dict(l=10, r=10, t=95, b=10),
                  title=dict(text="<b>Albatrosses rule the open sea; shearwaters and gulls stay near the coast; petrels are everywhere</b>"
                                  "<br><span style='font-size:13px;color:dimgrey'>How often each bird group was recorded in 10-minute counts from ships, 1969–1990. "
                                  "All four seasons are count equally.</span>",
                             x=0.01, font=dict(size=17)))
fig.show()

### Step 4: near the coast vs open sea
To put numbers on the pattern in the map, each coloured grid cell is given its distance to the nearest coastline and placed in a band: < 50 km, 50–150 km or > 150 km. The table gives each group's median rate in each band. Distances are approximate (1° ≈ 100 km at these latitudes).

In [ ]:
# Take the same rates, but split by distance from the coast
# For each surveyed grid cell, distance to land = distance from the cell's center to the nearest coastline
from shapely.geometry import box
coast = land.clip(box(140, -55, 185, -25)).union_all()
grid = gpd.GeoSeries(gpd.points_from_xy(*[a.ravel() for a in np.meshgrid(lon_mid, lat_mid)]))
km = grid.distance(coast).to_numpy().reshape(surveyed.shape) * 100   # 1° is about 100 km at these latitudes
band = pd.cut(km[surveyed], [0, 50, 150, np.inf], labels=["< 50 km", "50–150 km", "> 150 km"])
pd.DataFrame({g: rate[g][surveyed] for g in order}).groupby(band, observed=True).median().round(0).T

### Checks and limitations
* **Sensitivity.** We re-ran the map with a narrower and a wider blur (σ = 4 or 8 cells) and a lower and a higher effort threshold (3 or 12 counts per quarter). Albatrosses and petrels stay the most widespread groups, and prions stay low everywhere. The medians for shearwaters and gulls & terns rise under the strictest settings, because those keep mostly near-shore sea, where these groups are common. That is the same coastal pattern as in Step 4.
* **Data collection is concentrated around common shipping routes, and are not a random sample.** Ships followed fixed lanes, so the map describes the sea along those routes, not the whole region.
* **Years differ by area.** 76% of Tasman counts are from 1975–1984, while 89% of counts in New Zealand waters are from 1980–1989, so part of a difference between areas could be a difference between decades, ships or observers.
* **Data comes from observations, and do not depict true abundance.** Large birds and birds that follow ships are easier to spot than small ones, so rates compare how often a group was *recorded*.
* **Location is not independent from weather.** Wind and sea state vary from place to place, which overlaps slightly with the weather plot.